# 01 Exploratory Data Analysis
**AR Risk Scoring Project | Finance Analytics**

Goal: understand the synthetic dataset structure, distributions, class balance,
and relationships between features and the target variable (payment delinquency).


In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install",
                "pandas", "numpy", "matplotlib", "seaborn", "scipy", "-q"])


In [ ]:
import sqlite3, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
plt.rcParams["figure.dpi"] = 120

import sys
sys.path.append("../src")
from etl import run as run_etl

DB_PATH = Path("../data/processed/ar_risk.db")
if not DB_PATH.exists():
    print("Generating synthetic dataset...")
    run_etl()

with sqlite3.connect(DB_PATH) as conn:
    df = pd.read_sql("SELECT * FROM credit_transactions", conn)

print(f"Shape: {df.shape}")
df.head()


## 1. Dataset Overview

In [ ]:
df.info()


In [ ]:
df.describe().T.style.background_gradient(cmap="Blues", subset=["mean","std","50%"])


In [ ]:
# Missing values
missing = df.isnull().sum().sort_values(ascending=False)
missing = missing[missing > 0]
print("Missing values:")
print(missing.to_string())


## 2. Target Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Count plot
counts = df["serious_delinquency_2yrs"].value_counts()
axes[0].bar(["On Time (0)", "Default (1)"], counts.values,
            color=["#4C9BE8", "#E8674C"], edgecolor="white", linewidth=1.5)
axes[0].set_title("Class Distribution")
axes[0].set_ylabel("Count")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 500, f"{v:,}\n({v/len(df):.1%})", ha="center", fontsize=10)

# Pie
axes[1].pie(counts.values, labels=["On Time", "Default"],
            colors=["#4C9BE8", "#E8674C"], autopct="%1.1f%%",
            startangle=140, wedgeprops={"edgecolor": "white", "linewidth": 2})
axes[1].set_title("Default Rate")

plt.suptitle("Target Variable — Serious Delinquency", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()
print(f"\nClass imbalance ratio: 1 : {counts[0]/counts[1]:.1f}")


## 3. Feature Distributions

In [ ]:
num_cols = ["age", "monthly_income", "credit_limit", "debt_ratio",
            "num_open_credit_lines", "num_dependents"]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    data = df[col].dropna()
    # clip extreme values for readability
    p99 = data.quantile(0.99)
    data = data.clip(upper=p99)
    axes[i].hist(data, bins=40, color="#4C9BE8", edgecolor="white", alpha=0.85)
    axes[i].set_title(col.replace("_", " ").title())
    axes[i].set_xlabel("Value")
    axes[i].set_ylabel("Count")

plt.suptitle("Numerical Feature Distributions (clipped at 99th pct)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


## 4. Features vs Target

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    on_time  = df[df["serious_delinquency_2yrs"] == 0][col].dropna().clip(upper=df[col].quantile(0.99))
    default  = df[df["serious_delinquency_2yrs"] == 1][col].dropna().clip(upper=df[col].quantile(0.99))
    axes[i].hist(on_time, bins=35, alpha=0.6, color="#4C9BE8", label="On Time", density=True)
    axes[i].hist(default, bins=35, alpha=0.6, color="#E8674C", label="Default", density=True)
    axes[i].set_title(col.replace("_", " ").title())
    axes[i].legend(fontsize=8)

plt.suptitle("Feature Distribution by Target Class (density)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


## 5. Default Rate by Age Band

In [ ]:
df["age_band"] = pd.cut(df["age"], bins=[0,29,44,59,100],
                        labels=["Under 30","30-44","45-59","60+"])

age_stats = (df.groupby("age_band", observed=True)["serious_delinquency_2yrs"]
               .agg(["mean","count"]).rename(columns={"mean":"default_rate","count":"total"}))
age_stats["default_rate_pct"] = (age_stats["default_rate"] * 100).round(2)

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(age_stats.index.astype(str), age_stats["default_rate_pct"],
              color=["#4C9BE8","#5BA8D4","#E89B4C","#E8674C"], edgecolor="white")
for bar, val in zip(bars, age_stats["default_rate_pct"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f"{val:.1f}%", ha="center", fontsize=10, fontweight="bold")
ax.set_title("Default Rate by Age Band", fontsize=13, fontweight="bold")
ax.set_ylabel("Default Rate (%)")
ax.set_xlabel("Age Band")
plt.tight_layout()
plt.show()

print(age_stats[["total","default_rate_pct"]].to_string())


## 6. Late Payment Events vs Default Rate

In [ ]:
df["total_late_events"] = (df["num_late_30_59_days"] +
                           df["num_late_60_89_days"] +
                           df["num_times_90_days_late"])

late_stats = (df.groupby("total_late_events")["serious_delinquency_2yrs"]
                .agg(["mean","count"])
                .rename(columns={"mean":"default_rate","count":"customers"})
                .head(12))

fig, ax1 = plt.subplots(figsize=(10, 4))
ax2 = ax1.twinx()

ax1.bar(late_stats.index, late_stats["customers"], color="#CBD5E1", label="Customers")
ax2.plot(late_stats.index, late_stats["default_rate"]*100, color="#E8674C",
         marker="o", linewidth=2.5, label="Default Rate %")

ax1.set_xlabel("Total Late Payment Events")
ax1.set_ylabel("Number of Customers", color="#64748B")
ax2.set_ylabel("Default Rate (%)", color="#E8674C")
ax2.tick_params(axis="y", labelcolor="#E8674C")
ax1.set_title("Late Payments vs Default Rate", fontsize=13, fontweight="bold")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, labels1+labels2, loc="upper left")
plt.tight_layout()
plt.show()


## 7. Correlation Heatmap

In [ ]:
corr_cols = ["age","monthly_income","credit_limit","debt_ratio",
             "num_open_credit_lines","num_late_30_59_days",
             "num_late_60_89_days","num_times_90_days_late",
             "num_dependents","serious_delinquency_2yrs"]

corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, linewidths=0.5, ax=ax, annot_kws={"size":8})
ax.set_title("Feature Correlation Matrix", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


## 8. Key Findings

| Finding | Detail |
|---|---|
| Class imbalance | ~7–8% default rate — requires `class_weight='balanced'` in modeling |
| Strongest predictor | `num_times_90_days_late` — highest positive correlation with default |
| Age effect | Younger customers (<30) show slightly higher default rates |
| Debt ratio | Right-skewed; high values concentrated in default class |
| Missing data | ~8% in `monthly_income` — imputed with median in ETL |

**Next step → `02_feature_eng.ipynb`**
